In [1]:
#!/usr/bin/env python3
from __future__ import annotations

import re
import shutil
import subprocess
import sys
import unicodedata
from pathlib import Path
from datetime import datetime

The code below takes the articles in the rtf format and converts them to PDF creating the name with a unified format of date_title.

In [5]:
#Folder with original RTF files downloaded from Nexis Uni:
SOURCE_DIR = Path("/home/arabella/Desktop/Journals/taz_rtf/")
#Folder to write cleaned text files to:
TARGET_DIR = Path("/home/arabella/Desktop/Journals/taz_txt/")

SUPPORTED_EXTENSIONS = {".rtf"}

# Tolerant: "Load-Date" or "Load Date", any case, normal ":" or fullwidth "："
LOAD_DATE_LINE_RE = re.compile(r"(?im)^\s*Load[\s\-]*Date\s*[:：]\s*(.+?)\s*$")

# Parses: "August 13, 2024" (also "Aug 13 2024", optional comma, optional ordinal suffix)
MONTH_DAY_YEAR_RE = re.compile(
    r"^\s*([A-Za-z]+)\s+(\d{1,2})(?:st|nd|rd|th)?\s*,?\s*(\d{4})\s*$",
    re.IGNORECASE,
)


def should_skip(path: Path) -> bool:
    if path.suffix.lower() not in SUPPORTED_EXTENSIONS:
        return True
    if path.name.startswith("ad-hoc-"):
        return True
    return False


def slugify_filename(path: Path) -> str:
    """
    Turn filename stem into a stable slug:
      "Der Friedhof der Illusionen" -> "der-friedhof-der-illusionen"
    """
    s = path.stem.strip()

    # Normalize unicode (e.g., “ä” -> "ä"), then drop diacritics.
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))

    s = s.lower()

    # Replace separators with hyphen
    s = re.sub(r"[ _]+", "-", s)

    # Remove everything except alnum and hyphen
    s = re.sub(r"[^a-z0-9\-]+", "", s)

    # Collapse multiple hyphens and trim
    s = re.sub(r"-{2,}", "-", s).strip("-")

    if not s:
        raise ValueError(f"Could not slugify filename: {path.name}")
    return s


def rtf_to_text(path: Path) -> str:
    """
    Convert RTF to plain text.
    1) Try striprtf (pip install striprtf)
    2) Fall back to pandoc if installed
    """
    raw = path.read_text(encoding="utf-8", errors="ignore")

    try:
        from striprtf.striprtf import rtf_to_text as striprtf_to_text  # type: ignore
        return striprtf_to_text(raw)
    except Exception:
        pass

    try:
        proc = subprocess.run(
            ["pandoc", "-f", "rtf", "-t", "plain", str(path)],
            check=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
        )
        return proc.stdout
    except FileNotFoundError as e:
        raise RuntimeError(
            "RTF conversion failed: install 'striprtf' (recommended) or install 'pandoc'.\n"
            "  pip install striprtf\n"
            "or install pandoc from https://pandoc.org/installing.html"
        ) from e


def normalize_text(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    # normalize NBSP and narrow NBSP
    text = text.replace("\u00A0", " ").replace("\u202F", " ")

    lines = [ln.rstrip() for ln in text.split("\n")]
    text = "\n".join(lines)

    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip() + "\n"


def extract_date_prefix(text: str, *, path: Path) -> str:
    """
    Extract date from a line like: "Load-Date: August 13, 2024"
    Returns: YYYY_MM_DD
    """
    m = LOAD_DATE_LINE_RE.search(text)
    if not m:
        tail = "\n".join(text.splitlines()[-30:])
        raise ValueError(f"No Load-Date line found in {path.name}. Tail:\n{tail}")

    date_part = m.group(1).strip()

    # Handle wrapped dates like:
    # Load-Date: August 13,
    # 2024
    date_part = date_part.replace("\n", " ").strip()

    m2 = MONTH_DAY_YEAR_RE.match(date_part)
    if not m2:
        raise ValueError(f"Found Load-Date in {path.name} but couldn't parse date: {date_part!r}")

    month_name, day, year = m2.groups()

    # Full month name, then abbreviated
    try:
        dt = datetime.strptime(f"{month_name} {day} {year}", "%B %d %Y")
    except ValueError:
        dt = datetime.strptime(f"{month_name} {day} {year}", "%b %d %Y")

    return dt.strftime("%Y_%m_%d")


def main() -> None:
    TARGET_DIR.mkdir(exist_ok=True)

    converted = 0
    skipped_existing = 0
    skipped_ignored = 0
    skipped_unparseable = 0

    for source_path in sorted(SOURCE_DIR.iterdir()):
        if not source_path.is_file():
            continue

        if should_skip(source_path):
            skipped_ignored += 1
            continue

        try:
            slug = slugify_filename(source_path)
            text = normalize_text(rtf_to_text(source_path))
            date_prefix = extract_date_prefix(text, path=source_path)
        except Exception:
            skipped_unparseable += 1
            continue

        target_name = f"{date_prefix}_{slug}.txt"
        target_path = TARGET_DIR / target_name

        if target_path.exists():
            skipped_existing += 1
            continue

        target_path.write_text(text, encoding="utf-8")
        converted += 1

    print(f"Converted: {converted}")
    print(f"Skipped existing: {skipped_existing}")
    print(f"Skipped ignored: {skipped_ignored}")
    print(f"Skipped unparseable: {skipped_unparseable}")


if __name__ == "__main__":
    main()

Converted: 636
Skipped existing: 0
Skipped ignored: 0
Skipped unparseable: 0


The code below takes a folder directory with the created txt files and extracts the body of the article (getting rid of all the junk from uni lexus) and creates a new directory with the clean txt files.

In [2]:
import re
from pathlib import Path
#Folder with original text files exported from Spiegel:
IN_DIR = Path("/home/arabella/Desktop/Journals/bild_txt")
#Folder to write cleaned text files to:
OUT_DIR = Path("/home/arabella/Desktop/Journals/bild_txt_clean")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def extract_spiegel_body(raw_text: str) -> str:
    """
    Returns the article body for Spiegel exports:
    keeps text strictly between a line that is 'Body' and the line starting with 'Load-Date:'.
    Returns "" if markers are missing.
    """
    text = raw_text.replace("\r\n", "\n").replace("\r", "\n")

    # Start marker: line that is exactly "Body" (allow surrounding whitespace)
    start = re.search(r"(?im)^\s*Body\s*$", text)
    if not start:
        return ""

    # Everything after the marker line
    after = text[start.end():]

    # End marker: line starting with Load-Date:
    end = re.search(r"(?im)^\s*Load-Date:\s*.*$", after)
    body = after[: end.start()] if end else after

    # Cleanup: trim, collapse excessive blank lines, remove common footer
    body = body.strip()
    body = re.sub(r"\n{3,}", "\n\n", body)
    body = re.sub(r"(?im)^\s*End of Document\s*$", "", body).strip()

    return body

for fp in sorted(IN_DIR.glob("*.txt")):
    raw = fp.read_text(encoding="utf-8", errors="ignore")
    body = extract_spiegel_body(raw)

    # if you want to keep files even when parsing fails, you can write raw or skip
    if not body:
        continue

    (OUT_DIR / fp.name).write_text(body, encoding="utf-8")